# EDA — Tickets de Intercom como fuente de churnEl ML Canvas menciona los datos de contacto con soporte como features pendientes deincorporar, con la nota *"el equipo aún debe definir el método de obtención"*. Estenotebook evalúa el export de Intercom (`data/intercom/intercom_data.csv`) para respondertres preguntas:1. ¿Se puede vincular un ticket con una cuenta del snapshot?2. ¿Hay señal de churn en el contenido de los tickets?3. ¿Alcanza para entrenar hoy?**Respuesta corta: sí, sí y no.** La señal existe y es buena, pero este export enparticular no sirve todavía. El detalle, abajo.

In [ ]:
import sys, jsonsys.path.insert(0, "../src")import pandas as pdimport matplotlib.pyplot as pltRUTA = "../data/intercom/intercom_data.csv"pd.set_option("display.max_columns", 50)

## 1. Qué hay en el archivo

In [ ]:
# El archivo pesa 1 GB pero tiene pocas filas: cada ticket lleva la conversacion# completa embebida en `Ticket Parts`. Se lee por chunks y se aplana lo util.CAMPOS_ATTR = ["Motivo", "Tema", "Submotivo", "Necesita derivación",               "Nombre de cuenta", "ID de dash", "País"]def aplanar(ruta, chunksize=5000):    partes = []    for chunk in pd.read_csv(ruta, chunksize=chunksize, low_memory=False):        chunk.columns = [c.strip("\ufeff") for c in chunk.columns]        attrs = chunk["Ticket Attributes"].map(            lambda s: json.loads(s) if isinstance(s, str) else {}        )        out = pd.DataFrame({            "ticket_id": chunk["Ticket ID"],            "created_at": pd.to_datetime(chunk["Created At"], unit="s", errors="coerce"),            "category": chunk["Category"],            "channel": chunk["Channel"],            "company_id": chunk["Company ID"],            "tipo": chunk["Ticket Type"].map(                lambda s: json.loads(s).get("name") if isinstance(s, str) else None),        })        for campo in CAMPOS_ATTR:            out[campo] = attrs.map(lambda d, c=campo: d.get(c))        partes.append(out)    return pd.concat(partes, ignore_index=True)tk = aplanar(RUTA)tk["periodo"] = tk["created_at"].dt.strftime("%Y%m").astype(int)print(f"{len(tk):,} tickets · {tk['created_at'].min():%Y-%m-%d} a {tk['created_at'].max():%Y-%m-%d}")tk[["created_at", "tipo", "Tema", "Motivo", "ID de dash"]].head()

## 2. El archivo está truncado`20.000` filas exactas es un número redondo sospechoso, y el volumen mensual lo confirma:un mes con 12.114 tickets y otros con 7. Un SaaS con 32.000 cuentas activas no tiene esadistribución. Es el límite de paginación de la API de Intercom.

In [ ]:
por_mes = tk.groupby("periodo").size()fig, ax = plt.subplots(figsize=(11, 3.2))ax.bar(por_mes.index.astype(str), por_mes.values, color="#2f5bea", alpha=.85)ax.set_title("Tickets por mes — el export no es un histórico, es una ventana truncada")ax.tick_params(axis="x", rotation=90)ax.grid(alpha=.3, axis="y")plt.tight_layout()print(f"filas exactas: {len(tk):,}")print(f"tickets duplicados: {tk['ticket_id'].duplicated().sum()}")print(f"el mes pico concentra el {por_mes.max()/len(tk):.0%} de todo el archivo")

## 3. ¿Se puede vincular un ticket con una cuenta?

In [ ]:
tk["tiene_dash"] = tk["ID de dash"].notna() & tk["ID de dash"].astype(str).str.strip().ne("")tk["tiene_company"] = tk["company_id"].notna()cobertura = tk.groupby("tipo").agg(    tickets=("tiene_dash", "size"),    con_dash=("tiene_dash", "mean"),    con_company=("tiene_company", "mean"),)cobertura = cobertura[cobertura["tickets"] >= 20].sort_values("tickets", ascending=False)cobertura["con_dash"] = (cobertura["con_dash"] * 100).round(1)cobertura["con_company"] = (cobertura["con_company"] * 100).round(1)cobertura

Acá está el problema de fondo, y no es de volumen sino de captura:- Los tickets de **integraciones** traen el `ID de dash` en el **100%** de los casos: el  flujo lo pide porque lo necesita para operar.- Los de **soporte** (`Nueva Consulta`, `Consulta CS`, `Motivo Da Consulta`) lo traen en  **0-17%**. Y son justo los que interesan para churn.El `Company ID` de Intercom, en cambio, está en el 63% de los tickets. Ese es el puente:si se cruza con la tabla de Companies de Intercom (donde vive el atributo con el ID deFudo), se recupera casi todo.

In [ ]:
panel = pd.read_parquet("../outputs/interim/panel.parquet")dash = pd.to_numeric(tk.loc[tk["tiene_dash"], "ID de dash"], errors="coerce").dropna().astype(int)en_panel = set(dash) & set(panel["id"])print(f"cuentas identificables en los tickets: {dash.nunique():,}")print(f"de esas, presentes en el snapshot:     {len(en_panel):,} ({len(en_panel)/dash.nunique():.0%})")print(f"cobertura sobre la base activa:        {len(en_panel)/panel['id'].nunique():.1%}")print()print("El 'ID de dash' cruza limpio con el snapshot: la integración es viable.")

## 4. ¿Hay señal de churn en los tickets?

In [ ]:
# La taxonomia de motivos es rica y tiene categorias directamente ligadas a la baja.tk["Motivo"].value_counts().head(15)

In [ ]:
MOTIVOS_RIESGO = ["Solicitar la baja de mi cuenta", "Negociar mi plan/promoción"]for motivo in MOTIVOS_RIESGO:    s = tk[tk["Motivo"] == motivo]    print(f"{motivo!r}")    print(f"   {len(s):,} tickets · con ID de dash: {s['tiene_dash'].mean():.0%} "          f"· con company_id: {s['tiene_company'].mean():.0%}")

La señal es explícita — un ticket que dice *"Solicitar la baja de mi cuenta"* es churndeclarado — pero **ninguno de esos tickets trae el ID de la cuenta**. El 59% sí trae`company_id`, así que son recuperables con la tabla de Companies.Dos advertencias sobre esa señal, para cuando esté disponible:- **Cobertura baja.** En el único mes con volumen creíble (202604: 12.114 tickets) hay  155 tickets de baja, contra unas 1.200 bajas reales por mes. Cubre ~13%: la mayoría de  las cuentas se va sin pasar por soporte.- **Riesgo de leakage.** Si el ticket de baja se crea el mismo mes en que la cuenta  desaparece, no anticipa nada: es simultáneo al evento. Hay que usarlo con lag, igual que  el resto de las features.

## 5. ¿Alcanza para entrenar hoy?

In [ ]:
etiquetables = panel.loc[panel["churn"].notna(), "periodo"]rango_label = (etiquetables.min(), etiquetables.max())en_ventana = tk[tk["periodo"].between(*rango_label) & tk["tiene_dash"]]print(f"periodos con etiqueta de churn: {rango_label[0]} .. {rango_label[1]}")print(f"periodos con tickets:           {tk['periodo'].min()} .. {tk['periodo'].max()}")print()print(f"tickets identificables que caen en periodos etiquetables: {len(en_ventana)}")print(f"cuentas afectadas: {en_ventana['ID de dash'].nunique()}")print()print("No: con esa cantidad no se puede medir ningún efecto.")

## Conclusiones**La hipótesis es buena y la fuente sirve**, pero este export no permite probarla. Tresproblemas, en orden de importancia:| Problema | Detalle | Se arregla ||---|---|---|| **Los tickets de soporte no registran la cuenta** | 0% de los tickets de baja tienen `ID de dash`; los de integraciones tienen 100% | Cruzando por `Company ID` con la tabla de Companies de Intercom || **Export truncado** | 20.000 filas exactas, corta el 2026-05-28 | Re-exportar paginando completo || **Ventana temporal desalineada** | Tickets desde 2025-07; las etiquetas de churn llegan hasta 2025-12 | Pedir desde 2024-01 |### Qué pedir en el re-export1. **Sin límite de registros**, desde **2024-01** — que cubra el panel de churn.2. **La tabla de Companies de Intercom**, con el atributo que guarda el ID de Fudo. Es lo   que permite atribuir los tickets de soporte, que son los que importan.3. **Sin la columna `Ticket Parts`.** Son el 99% del peso del archivo (1 GB para 20.000   tickets) y el modelo no las necesita: alcanzan los metadatos. Sin ellas, el mismo   volumen entra en pocos MB y se puede exportar el histórico completo.Campos que sí hacen falta: `Ticket ID`, `Created At`, `Updated At`, `Company ID`,`Ticket Type`, `Ticket State`, `Category`, `Channel`, y de `Ticket Attributes` los campos`ID de dash`, `Tema`, `Motivo`, `Submotivo` y `Necesita derivación`.### Features que se podrían construirCon los datos completos, siguiendo el mismo criterio que el resto del pipeline —elmovimiento importa más que el nivel— y siempre con lag para no filtrar el futuro:- `tickets_1m`, `tickets_3m`, y su variación contra el promedio móvil.- `meses_desde_ultimo_ticket`.- Conteo por `Tema` en las categorías más frecuentes (facturación, impresoras, ventas).- `tickets_de_baja_declarada` y `tickets_de_negociacion_de_plan` (con lag ≥ 1 mes).- `tickets_derivados` — un escalamiento a Ventas/CS es señal de fricción comercial.- `tickets_sin_resolver` sobre el total.